# W_u arms — non-zero init / own learning rate, **train on Nexar, eval on DoTA-CAP-dev**, 1 seed

Rule (written before any number): `core/docs/final/WU_ARMS.md`. Host = v2 Nexar A2's recipe (`W_u` zero-init, AdamW 5e-5),
SA² and the prefix **off** — no patch cache, no frame archive.

| arm | `W_u` init | LR on `W_u` | config |
|---|---|---|---|
| **W0** (control, trained here) | 0 | 5e-5 | — |
| **WI** | N(0, 0.003²) | 5e-5 | `model.motion_init_std=0.003` |
| **WL** | 0 | 5e-4 | `train.motion_lr_mult=10` |
| WIL (optional, add to `ARMS`) | N(0, 0.003²) | 5e-4 | both |

Every contrast is arm − W0; R1 s2024 (0.8102, v5 Stage 2) is printed beside it. One seed ⇒ every sentence is
**PRELIMINARY (n = 1 seed)** and adopts nothing.

| step | what | GPU? |
|---|---|---|
| 1–3 | setup, deps, helpers | — |
| 4 | **train** `ARMS` × s2024 (1,749 steps) + config provenance vs v2 A2 | **yes** |
| 5 | protocol B on DoTA-CAP-dev per arm | yes |
| 6 | **read-out** → paste `REPORTS/wu_readout.md` to Claude | no |

**Drive:** code `Thesis-Final/kat-vad` (upload branch `v_final`); outputs under `Thesis-Final/outputs/wu/`.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data (Nexar raw, DoTA, CAP tars), cache/clip
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: Nexar_window, A2 bakes, KNN, census, v2 A2 runs, alignment
DRIVE_V5 = '/content/drive/MyDrive/Thesis-V5'     # read-only: R1's DoTA-CAP-dev read (v5 Stage 2)
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code + every cache / output this notebook writes
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'     # new caches (clip_patches/, clip_windows/) land here
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'WU_ARMS'), 'this upload predates the W_u arms -- re-upload kat-vad/'
assert (REPO / 'core/docs/final/WU_ARMS.md').is_file(), 'no WU_ARMS.md (the read-out rule) in this upload'

ARMS     = ['W0', 'WI', 'WL']     # add 'WIL' for both knobs; W0 (the control) must stay first
SEEDS    = [constants.WU_SEEDS[0]]   # s2024 -> PRELIMINARY (n = 1 seed). list(constants.WU_SEEDS) = full read
CONTROL  = constants.WU_CONTROL_ARM   # W0
R1_NAME  = constants.V5_CONTROL_ROW   # printed beside W0 (reproduction of the host), never the control
ENC      = constants.V5_ENCODER       # V2-S
C        = constants.NEXAR_CONSTRUCTION_WINDOW
DATASET  = constants.NEXAR_DATASET
CAP_DEV  = constants.V2_SPLIT_DOTA_CAP_DEV
assert all(a in constants.WU_ARMS for a in ARMS) and ARMS[0] == CONTROL, ARMS

V2_CACHE, V2_OUT = Path(DRIVE_V2) / 'cache', Path(DRIVE_V2) / 'outputs'
BUILD    = V2_OUT / 'v2' / 'REPORTS' / 'nexar_n4' / 'build_report.json'
DATA     = Path(DRIVE_V2) / 'data' / f'{DATASET}_{C}'
A2_BAKE  = V2_CACHE / 'v2' / 'A2_S' / 'Nexar_s8'                         # host rows [x ; c u~/sigma_u]
KNN      = V2_CACHE / 'knn' / f'{DATASET}_{C}' / constants.KNN_CACHE_FILENAME
A2_RUNS  = V2_OUT / constants.NEXAR_FULL_RUNS_DIR / 'A2'                  # config provenance
DOTA     = constants.DATA_ROOT / 'DoTA'
DOTA_S1  = Path(DRIVE) / 'cache' / 'clip' / 'DoTA_s1_ncc'
DOTA_A2  = V2_CACHE / 'v2' / 'A2_S' / f'{CAP_DEV}_s3_from_s1_nexar'
R1_PB    = Path(DRIVE_V5) / 'outputs' / 'v5' / 'v5_stage2' / 'eval' / 'dota' / CAP_DEV / R1_NAME
A2_PB    = V2_OUT / constants.NEXAR_FULL_RUNS_DIR / 'eval' / 'dota' / 'A2' / CAP_DEV / 'protocol_b_readout.json'

ROOT     = constants.OUTPUT_ROOT / constants.WU_OUTPUT_DIRNAME
REPORTS  = ROOT / 'REPORTS'
LOCAL    = Path('/content/wu')
for p in (REPORTS, LOCAL):
    p.mkdir(parents=True, exist_ok=True)

PLAN = json.loads(BUILD.read_text())['plan'][C]
STEPS, EPOCHS = PLAN['steps'], PLAN['epochs']     # 1,749 = 33 x 53
for name, path in (('Nexar_window', DATA), ('A2 bake', A2_BAKE), ('KNN', KNN), ('v2 A2 runs', A2_RUNS),
                   ('DoTA s1', DOTA_S1), ('DoTA-CAP-dev A2 bake', DOTA_A2), ('DoTA labels', DOTA / 'labels_s8'),
                   ('R1 read', R1_PB / 'clip_aucs.json')):
    assert path.exists(), f'missing {name}: {path}'
    print(f'  {name:22s} {path}')
print(f'ARMS {ARMS} | SEEDS {SEEDS} | steps {STEPS} ({EPOCHS} epochs)')
print('runs/reads ->', ROOT)


In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn xlrd
python -c "import av, einops, faiss, sklearn; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 4-7 need one'
df -h /content | tail -1

In [ ]:
import pickle
import shutil
import subprocess
import time

import numpy as np
import torch
import yaml

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF (absl eats INFO logs)


def run(cmd, log=None, quiet_steps=False):
    """Stream a child's output here (and to ``log``); raise with its tail on failure."""
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                if not (quiet_steps and 'epoch' in line and 'batch' in line):
                    print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    """Copy a Drive dir (or only ``ids`` + manifests) / file to VM disk once (bit-identical; C2)."""
    if dst.exists() and ids is None:
        return dst
    t0 = time.time()
    if src.is_dir() and ids is None:
        tmp = dst.with_name(dst.name + '.partial')
        shutil.rmtree(tmp, ignore_errors=True)
        shutil.copytree(src, tmp)
        os.replace(tmp, dst)
    elif src.is_dir():
        dst.mkdir(parents=True, exist_ok=True)
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for pattern in ('*.json', '*.npz'):
            for m in src.glob(pattern):
                shutil.copy2(m, dst / m.name)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        tmp = dst.with_name(dst.name + '.partial')
        shutil.copy2(src, tmp)
        os.replace(tmp, dst)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    """VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10, lesson (bi))."""
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def ckpt_step(path):
    """The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step)."""
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path}: {exc}')
        return -1


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None


def flush_drive():
    t0 = time.time()
    drive.flush_and_unmount()
    drive.mount('/content/drive')
    print(f'Drive flushed and remounted in {time.time() - t0:.0f}s')


def write_ids(path, ids):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(f'{v}\n' for v in ids))
    return path


from core.data.v2_splits import load_split    # noqa: E402

CAP_IDS = sorted(load_split(CAP_DEV))
assert len(CAP_IDS) == 569, len(CAP_IDS)
print(f'{len(CAP_IDS)} DoTA-CAP-dev clips')

## 4. Train `ARMS` × `SEEDS` (GPU), then config provenance

Exactly the STP notebook's host command (v2 Nexar A2's recipe, SA² and prefix off) plus the arm's `model.motion_init_std` / `train.motion_lr_mult`. Local training, atomic publish to Drive, J10 re-verified after a flush, then `verified.json`; a verified run is never retrained.


In [ ]:
INPUTS = {'data': (DATA, LOCAL / 'stage' / f'data_{C}'), 'a2': (A2_BAKE, LOCAL / 'stage' / 'A2_s8'),
          'knn': (KNN, LOCAL / 'stage' / f'knn_{C}.npz')}
_STAGED = {}


def staged(key):
    if key not in _STAGED:
        _STAGED[key] = stage(*INPUTS[key])
    return _STAGED[key]


def arm_sets(arm):
    return [x for k, v in constants.WU_ARMS[arm].items() for x in ('--set', f'{k}={v}')]


def train_cmd(arm, seed, out_dir):
    cmd = [sys.executable, '-m', 'core.train',
           '--set', 'train.stage=2', '--set', f'train.seed={seed}',
           '--set', f'train.num_epochs={EPOCHS}', '--set', 'train.amp=true',
           '--set', f'data.dataset={DATASET}',
           '--set', f'model.score_head_kernel={constants.DADA_ORIGIN_SCORE_HEAD_KERNEL}',
           '--set', f'loss.mil_topk_pct={constants.DADA_ORIGIN_MIL_TOPK_PCT}',
           '--set', 'kip.enabled=false', '--set', 'loss.dvs_anchor_mode=span',
           '--set', 'data.is_egocentric=false',
           '--set', f'v2.motion={ENC}', '--set', f'model.motion_dim={constants.V5_GRID_DIM}', *arm_sets(arm),
           '--data-dir', staged('data'), '--clip-dir', staged('a2'), '--knn-cache', staged('knn'),
           '--output-dir', out_dir]
    return cmd


def is_verified(drive_dir):
    rec, ckpt = drive_dir / 'verified.json', drive_dir / 'checkpoint_last.pt'
    if rec.exists() and ckpt.exists():
        r = json.loads(rec.read_text())
        return r['step'] == STEPS and r['ckpt_bytes'] == ckpt.stat().st_size
    return False


def train_arm(arm, seed):
    drive_dir, local = ROOT / arm / f's{seed}', LOCAL / 'runs' / arm / f's{seed}'
    if is_verified(drive_dir):
        print(f'{arm} s{seed}: verified on Drive -- not retrained')
        return
    local.mkdir(parents=True, exist_ok=True)
    resume = ['--resume', local / 'checkpoint_last.pt'] if (local / 'checkpoint_last.pt').exists() else []
    t0 = time.time()
    run(train_cmd(arm, seed, local) + resume, log=LOCAL / f'train_{arm}_s{seed}.log', quiet_steps=True)
    last = metrics_last(local / 'metrics.jsonl')
    assert ckpt_step(local / 'checkpoint_last.pt') == last['global_step'] == STEPS, f'{arm} s{seed}: J10 locally'
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
        publish(local / name, drive_dir / name)
    flush_drive()
    assert ckpt_step(drive_dir / 'checkpoint_last.pt') == STEPS, f'{arm} s{seed}: Drive copy is not step {STEPS}'
    (LOCAL / 'verified.json').write_text(json.dumps(
        {'step': STEPS, 'ckpt_bytes': (drive_dir / 'checkpoint_last.pt').stat().st_size}))
    publish(LOCAL / 'verified.json', drive_dir / 'verified.json')
    print(f'{arm} s{seed}: {(time.time() - t0) / 60:.0f} min | last mil {last.get("mil"):.4f} | '
          f'rho_u {last.get("motion_share")} | |W_u| {last.get("w_u_norm")}')


assert torch.cuda.is_available(), 'torch sees no GPU -- train.device=auto would silently train on CPU'
for seed in SEEDS:
    for arm in ARMS:
        train_arm(arm, seed)

# provenance: config == v2 Nexar A2's of the same seed except the keys v2 predates (`stp.*`, `v5.*`) and the two
# W_u knobs, which must carry exactly the arm's values (WU_ARMS.md §2)
from core.tools.v5_stage1 import _flat

KNOBS = ('model.motion_init_std', 'train.motion_lr_mult')
DEFAULTS = {'model.motion_init_std': 0.0, 'train.motion_lr_mult': 1.0}
for seed in SEEDS:
    ref = _flat(yaml.safe_load((A2_RUNS / f's{seed}' / 'config.yaml').read_text()))
    for arm in ARMS:
        got = _flat(yaml.safe_load((ROOT / arm / f's{seed}' / 'config.yaml').read_text()))
        diff = {k: (ref.get(k), got.get(k)) for k in set(ref) | set(got)
                if not k.startswith(('stp.', 'v5.')) and k not in KNOBS and ref.get(k) != got.get(k)}
        assert not diff, f'{arm} s{seed} differs from v2 A2 beyond the W_u knobs: {diff}'
        assert got['v5.readout'] == constants.V5_READOUT_OFF and got['stp.sa2'] is False, (got['v5.readout'], got['stp.sa2'])
        want = {**DEFAULTS, **constants.WU_ARMS[arm]}
        assert all(got[k] == v for k, v in want.items()), (arm, {k: got[k] for k in want})
        print(f'{arm} s{seed}: config == v2 A2 s{seed} except', {k: got[k] for k in KNOBS})


## 5. DoTA-CAP-dev — protocol B per arm (GPU)

`s1[::3]`, whole clip, per-clip min-max, native frames, macro over two-class clips. Input = v2's A2 bake of CAP-dev. Skipped when the arm's read already covers `SEEDS`. Prints W0 vs R1 (host reproduction).


In [ ]:
PB_OUT = ROOT / 'eval' / 'dota' / CAP_DEV
D_INPUTS = {'dota_a2': (DOTA_A2, LOCAL / 'stage' / 'dota_a2', CAP_IDS),
            'dota_s1': (DOTA_S1, LOCAL / 'stage' / 'dota_s1', CAP_IDS),
            'dota_lab': (DOTA / 'labels_s8', LOCAL / 'stage' / 'dota_labels', None)}


def d_staged(key):
    if key not in _STAGED:
        _STAGED[key] = stage(*D_INPUTS[key])
    return _STAGED[key]


def read_covers(path):
    if not path.exists():
        return False
    return sorted(json.loads(path.read_text()).get('per_run_macro', {})) == sorted(f's{s}' for s in SEEDS)


wrote = False
for arm in ARMS:
    out = PB_OUT / arm
    if read_covers(out / 'protocol_b_readout.json'):
        print(f'dota {arm}: already read on seeds {SEEDS}')
        continue
    local, t0 = LOCAL / 'dota' / arm, time.time()
    runs = [x for s in SEEDS for x in ('--run', f's{s}', ROOT / arm / f's{s}' / 'checkpoint_last.pt')]
    run([sys.executable, '-m', 'core.tools.protocol_b_eval', *runs,
         '--input-dir', d_staged('dota_a2'), '--s1-dir', d_staged('dota_s1'),
         '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
         '--data-dir', d_staged('dota_lab'), '--split', CAP_DEV, '--out-dir', local, '--device', 'auto'],
        log=LOCAL / f'dota_{arm}.log')
    publish(local, out)
    wrote = True
    r = json.loads((local / 'protocol_b_readout.json').read_text())
    print(f"dota {arm}: macro {r['macro']['mean']:.4f} per seed {r['per_run_macro']} ({(time.time() - t0) / 60:.1f} min)")
if wrote:
    flush_drive()
r1 = json.loads((R1_PB / 'protocol_b_readout.json').read_text())
print(f"printed R1 (v5 Stage 2 read): macro {r1['macro']['mean']:.4f} per seed {r1['per_run_macro']}")
w0 = json.loads((PB_OUT / CONTROL / 'protocol_b_readout.json').read_text())
gap = abs(w0['macro']['mean'] - r1['macro']['mean'])
print(f"W0 {w0['macro']['mean']:.4f} vs R1 {r1['macro']['mean']:.4f}: |gap| {gap:.4f}" + ('  -- host does NOT reproduce R1 (> 0.01), flag only' if gap > 0.01 else ''))
if A2_PB.exists():
    print('printed: v2 Nexar A2 per seed', json.loads(A2_PB.read_text()).get('per_run_macro'))

## 6. Read-out → **paste `REPORTS/wu_readout.md` to Claude**

Rule: `WU_ARMS.md` §3. CPU only. Contrasts are arm − W0; the row `R1 - W0` is the host's reproduction, not a W_u effect. With one seed every sentence reads **PRELIMINARY (n = 1 seed)**.


In [ ]:
arms = [x for a in ARMS[1:] for x in ('--arm', a, PB_OUT / a)] + ['--arm', R1_NAME, R1_PB]   # 'R1 - W0' = the host's reproduction
run([sys.executable, '-m', 'core.tools.stp_readout', '--control', CONTROL, PB_OUT / CONTROL, *arms,
     '--seeds', *SEEDS, '--runs-root', ROOT, '--metadata', DOTA / 'metadata_val.json',
     '--split-file', DOTA / 'val_split.txt', '--out-dir', REPORTS, '--name', 'wu_readout',
     '--title', 'W_u arms read-out', '--rule', 'core/docs/final/WU_ARMS.md §3',
     '--control-note', '(same code, W_u zero-init, LR x1)'])
print((REPORTS / 'wu_readout.md').read_text())
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print('Drive flushed -- paste REPORTS/wu_readout.md to Claude')
